# Chapter 5 — Methods and Methodologies
### Notebook 2 · Ontology quality: OntoClean

*Book reference: Section 5.2*

A reasoner checks consistency. OntoClean checks whether your taxonomy makes ontological sense. The gap between those two is where most real modelling errors live — and every example in this notebook is **consistent**.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch05_toolkit as ch5
from oe_course.sparql import SparqlStore
from oe_course.data import corpus
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. Meta-properties

OntoClean tags each class with four meta-properties:

| Meta-property | Values | Reading |
|---|---|---|
| **rigidity** | `+R` / `~R` / `-R` | is the property essential to its instances? |
| **identity** | `+I` / `-I` | does it carry a criterion for 'the same one'? |
| **unity** | `+U` / `~U` / `-U` | are its instances wholes? |
| **dependence** | `+D` / `-D` | does it depend on something external? |

The key one is **rigidity**. `Person` is rigid: nothing stops being a person while continuing to exist. `Student` is **anti-rigid**: every student can stop being one. That difference is invisible to a DL reasoner and decisive here.

In [ ]:
rows = [{'class': name, 'rigidity': t.rigidity, 'identity': t.identity,
         'unity': t.unity, 'dependence': t.dependence}
        for name, t in ch5.ONTOCLEAN_TAGS.items()]
print(pd.DataFrame(rows).to_string(index=False))

## 2. The taxonomy constraints

The meta-properties constrain what may subsume what:

In [ ]:
for c in ch5.ONTOCLEAN_CONSTRAINTS:
    print(f"[{c['id']}]")
    print(f"  {c['statement']}")
    print(f"  why: {c['why']}\n")

## 3. Auditing a taxonomy

Here is a small taxonomy. Read it first and try to spot the errors by eye — then run the checker.

In [ ]:
for sub, sup in ch5.TAXONOMY:
    print(f'  {sub} <= {sup}')

In [ ]:
violations = ch5.ontoclean_violations()
for v in violations:
    print(f"[{v['constraint']}]")
    print(f"  axiom : {v['axiom']}")
    print(f"  detail: {v['detail']}\n")
print(f'{len({v["axiom"] for v in violations})} offending axioms, '
      f'{len(violations)} constraint breaches')

> Note `Person <= Student` breaches **two** constraints at once (rigidity *and* dependence). Real modelling errors usually violate several principles — which is why they feel wrong long before you can say why.

## 4. The point: a reasoner sees nothing wrong

Every axiom above is logically consistent. Let's prove it with the Chapter 3 tableau — the same reasoner that caught `Giraffe ⊓ Carnivore` without complaint here.

In [ ]:
sys.path.insert(0, str(Path.cwd().parent / 'ch03_description_logics'))
import ch03_toolkit as dl

tbox = dl.TBox()
for sub, sup in ch5.TAXONOMY:
    tbox.add(dl.Atomic(sub), dl.Atomic(sup))

for name in ['Person', 'Student', 'Statue', 'Pet']:
    ok = dl.satisfiable(dl.Atomic(name), tbox).satisfiable
    print(f'  {name:10s} satisfiable: {ok}')
print('\nEvery class is satisfiable and the KB is consistent. The reasoner has\n'
      'no complaint. The errors are real all the same -- they are about what\n'
      'the classes MEAN, and meaning is not a logical property.')

In [ ]:
print('what the reasoner infers from Person <= Student:')
print('  Person <= Person? ', dl.subsumes(dl.Atomic('Person'), dl.Atomic('Person'), tbox))
print('  Person <= Entity? ', dl.subsumes(dl.Atomic('Person'), dl.Atomic('Entity'), tbox))
print('\nIt cheerfully propagates the bad axiom. A reasoner amplifies whatever\n'
      'you assert -- including your mistakes. That is the argument for §5.2\n'
      'existing as a separate activity from §3.3.')

### Exercise 2.1 — Tag a new class and predict the violation

`Patient` is a role: nothing is essentially a patient. Tag it, add `Person <= Patient` to the taxonomy, and confirm the checker flags it. Then add `Patient <= Person` instead and confirm it does not.

> **Hint.** Roles are anti-rigid (`~R`) and externally dependent (`+D`).

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 2.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
tags = dict(ch5.ONTOCLEAN_TAGS)
tags['Patient'] = ch5.MetaProperties('~R', '-I', '-U', '+D')

wrong = ch5.ontoclean_violations([('Person', 'Patient')], tags)
right = ch5.ontoclean_violations([('Patient', 'Person')], tags)
print('Person <= Patient ->', [v['constraint'] for v in wrong])
print('Patient <= Person ->', [v['constraint'] for v in right] or 'no violation')
assert wrong and not right
print('\nThe direction is everything. A role may be subsumed BY a rigid class;\n'
      'it may never subsume one. Every taxonomy that puts a role above a natural\n'
      'kind has this bug, and it is one of the most common real-world errors.')

### Exercise 2.2 — Repair the taxonomy

Fix all three offending axioms in `TAXONOMY` without deleting any class, and confirm the checker reports nothing. State the modelling claim each repair makes.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 2.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
repaired = [
    ('Person', 'Entity'),
    ('Animal', 'Entity'),
    ('PhysicalObject', 'Entity'),
    ('Student', 'Person'),        # was Person <= Student: direction reversed
    ('Employee', 'Person'),
    ('Statue', 'PhysicalObject'), # was Statue <= Clay: constitution, not subsumption
    ('Pet', 'Animal'),            # was Person <= Pet: a pet is a role on an animal
]
print('violations after repair:', ch5.ontoclean_violations(repaired) or 'none')
assert ch5.ontoclean_violations(repaired) == []
print('\nEach repair is a claim:\n'
      '  1. students are a kind of person, not the reverse;\n'
      '  2. a statue is CONSTITUTED OF clay, not a KIND OF clay -- constitution\n'
      '     is a different relation, which is exactly Chapter 6 material;\n'
      '  3. being a pet is a role an animal plays.\n'
      'Note that repair 2 could not be expressed as subsumption at all. OntoClean\n'
      'found the error; fixing it needed a richer relation vocabulary.')